In [ ]:
import os
import pickle
import seaborn as sns

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, classification_report, roc_auc_score, precision_recall_curve, auc, \
    confusion_matrix, matthews_corrcoef
from collections import defaultdict

In [ ]:
# 对CRC owen459的数据对比其他方法画一下recall的热图
def calculate_recall(group_df, threshold=0.5):
    """计算recall: TP / (TP + FN)"""
    if len(group_df) == 0:
        return np.nan

    # 真实阳性
    true_positives = pd.Series([True] * len(group_df), index=group_df.index)
    # true_positives = group_df['label'] == 1
    # 预测阳性 (概率 > 阈值)
    pred_positives = group_df['pred'] == 1

    tp = ((true_positives) & (pred_positives)).sum()  # 真阳性
    fn = ((true_positives) & (~pred_positives)).sum()  # 假阴性

    if (tp + fn) == 0:
        return np.nan
    return tp / (tp + fn)




def get_group_recall(subpheno_df, dir, filename, order, object, index_col):
    split_5_recall = {}
    for i in range(1, 6):
        split = 'split'+str(i)

        prob_df = pd.read_csv(os.path.join(dir, split, filename),index_col=index_col)
        prob_459_df = prob_df.loc[list(subpheno_df.index),:]
        prob_459_df[object] = subpheno_df[object]

        location_results = prob_459_df.groupby(object).apply(calculate_recall).reset_index()
        location_results.columns = ['Group', 'Recall']
        location_results['Group'] = pd.Categorical(
            location_results['Group'],
            categories=order,
            ordered=True
        )
        location_results_sorted = location_results.sort_values('Group')
        print(location_results_sorted)
        split_5_recall[split]=location_results_sorted['Recall'].values.tolist()

    split_5_recall_df = pd.DataFrame(dict(split_5_recall), index=order)
    split_5_recall_df['mean'] = split_5_recall_df.mean(1)
    group_counts = subpheno_df[object].value_counts()
    return split_5_recall_df, group_counts

def create_and_plot_heatmap(df, group_counts, object):
    """完整的热图创建和绘制流程"""

    plt.style.use('default')
    plt.rcParams.update({'font.size': 22})
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['pdf.fonttype'] = 42

    # 3. 绘制
    fig, ax = plt.subplots(figsize=(8,8)) # (8,8) tumor

    sns.heatmap(df,
                annot=True,
                fmt=".2f",
                cmap="YlGnBu",
                square=True,
                ax=ax,
                annot_kws={"size": 20},
                cbar_kws={"shrink": 0.8},
                linewidths=0.8,
                linecolor='white'
                )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    # ax1.set_xticklabels([])
    yticklabels = [t.get_text() for t in ax.get_yticklabels()]
    new_labels = [f"Stage {label} (n={group_counts.loc[label]})" for label in yticklabels if label != 'Average']+['Average']

    ax.set_yticks(ax.get_yticks())  # 保持原有刻度位置
    ax.set_yticklabels(new_labels, rotation=0, fontproperties={'family': 'Arial', 'size': 22})
    plt.tight_layout()

    # 4. 保存
    # plt.savefig(f'/home/share/huadjyin/home/zhangkexin2/code/meta_index/experiment/figures/finetune/contrast/owen459.recall.{object}.contrast.pdf', dpi=800, bbox_inches='tight')
    plt.show()

In [ ]:
def create_and_plot_heatmap(df, group_counts, object_name):
    """
    重构后的顶刊风格热图函数
    改动点：
    1. 弃用 sns.heatmap，改用 ax.imshow 实现更精准的像素控制。
    2. 增加 secondary_xaxis，在顶部显示分类标签 (如 Stage, Location)。
    3. 统一命名：将 'Average' 改为更学术的 'Overall'。
    4. 增强网格感：使用白色厚网格线 (linewidth=4) 营造“瓷砖”质感。
    5. 自适应文字颜色：根据背景深浅自动切换黑/白字。
    """
    # --- 配置学术环境 ---
    plt.rcParams['font.family'] = 'Arial'
    plt.rc('font', size=22)
    plt.rcParams['pdf.fonttype'] = 42
    # 准备数据
    # 改动：将 Index 中的 'Average' 替换为 'Overall'
    df = df.rename(index={'Average': 'Overall'})
    data_values = df.values
    rows = df.index.tolist()
    cols = df.columns.tolist()
    # 创建画布
    fig, ax = plt.subplots(figsize=(10, 10), facecolor='white') #(10,10)
    # --- 绘图核心 ---
    # 改动：使用 RdYlBu_r 配色，更具视觉冲击力和学术辨识度
    im = ax.imshow(data_values, cmap='RdYlBu_r', aspect='auto',vmin=0, vmax=1)
    # --- 坐标轴美化 ---
    ax.set_xticks(np.arange(len(cols)))
    ax.set_yticks(np.arange(len(rows)))
    # 改动：底部 X 轴 45 度旋转，ha="right" 确保末端对齐
    ax.set_xticklabels(cols, rotation=45, ha="right", rotation_mode="anchor")
    # 改动：Y 轴集成样本量信息，并对 Overall 加粗处理
    new_yticklabels = []
    for label in rows:
        if label == 'Overall':
            new_yticklabels.append(r'$\mathbf{Overall}$')  # 使用 LaTeX 加粗
        else:
            n_count = group_counts.get(label, "N/A")
            new_yticklabels.append(f"{label} (n={n_count})")
    ax.set_yticklabels(new_yticklabels)

    # --- 数值标注 ---
    # 改动：增加阈值判断，深色格子显示白字，浅色显示黑字
    threshold = (data_values.max() + data_values.min()) / 2
    for i in range(len(rows)):
        for j in range(len(cols)):
            val = data_values[i, j]
            # 根据背景色深浅切换字体颜色
            text_color = "white" if val > 0.7 else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", color=text_color)
    # --- 细节打磨 ---
    # 改动：隐藏外边框，绘制白色加厚网格线
    ax.spines[:].set_visible(False)
    ax.set_xticks(np.arange(len(cols) + 1) - .5, minor=True)
    ax.set_yticks(np.arange(len(rows) + 1) - .5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=4)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(bottom=False, left=False)
    # 改动：颜色条微调，移除外边框，增加 labelpad
    cbar = fig.colorbar(im, ax=ax, pad=0.03, shrink=0.8)
    cbar.outline.set_visible(False)
    cbar.set_label('Recall', rotation=-90, va="bottom", labelpad=20)
    plt.tight_layout()
    plt.savefig(f'/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/figures/finetune/11.25/contrast/owen_Recall_{object_name}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
subpheno_df = pd.read_csv(
    '/bgi-seq-model-2/datasets/zhangkexin/meta_index/metaphlan4/validate/7.25/CRC/owen/CRC_colon_phenotype_Qin.txt',
    sep='\t', index_col=0)
subpheno_df = subpheno_df.drop('T2104103043')


# 肿瘤位置
contrast_group_recall={}
order=['Rectum', 'Sigmoid colon', 'Ascending colon', 'Transverse colon', 'Descending colon', 'Ileoceum', 'Rectosigmoid junction']
MLP_split5_recall_df, _ = get_group_recall(subpheno_df, '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/MLP-model/indep.test/12.1/pandisease/', 'owen_profile_mlp_probs.csv', order,'Tumor_Location', 1 )
contrast_group_recall['MLP']=MLP_split5_recall_df['mean'].tolist()

for method in ['lr', 'rf', 'xgb']:
    dir = '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/ML-model/indep.test/11.25/pandisease/'
    split_5_recall_df, _ = get_group_recall(subpheno_df, dir,f'owen_profile_{method}_probs.csv', order, 'Tumor_Location', 1)
    contrast_group_recall[method] = split_5_recall_df['mean'].tolist()

metaGPT_split5_recall_df, group_counts = get_group_recall(subpheno_df, '/bgi-seq-model-2/codes/zhangkexin/meta_index/output/llama/v4/PanDisease/Split5.specific.aug.Full.sortabu.ema.12.4/',
                                                       'best_ckpt/result/probs/metaGPT.crc.owen.test.prob.csv',
                                                       order, 'Tumor_Location', 0)

contrast_group_recall['UmetaGPT']=metaGPT_split5_recall_df['mean'].tolist()
contrast_df = pd.DataFrame(contrast_group_recall, index=order, columns=['MLP', 'lr', 'rf', 'xgb', 'UmetaGPT'])
contrast_df.columns = ['MLP', 'LR', 'RF', 'XGBoost', 'UmetaGPT']
# contrast_df.loc['Average'] = contrast_df.mean(0)
contrast_df.loc['Overall'] = [0.7804, 0.8087, 0.7521, 0.8366, 0.8444]

create_and_plot_heatmap(contrast_df, group_counts, 'Tumor_Location')


# 疾病分期
subpheno_df_filter = subpheno_df.dropna(subset=['cStage'])
map_stage = {2.0: 'II', 3.0: 'III', 1.0: 'I', 4.0: 'IV'}
subpheno_df_filter['cStage'] = [map_stage[i] for i in subpheno_df_filter['cStage']]

contrast_group_recall={}
order=['I', 'II', 'III', 'IV']
MLP_split5_recall_df, _ = get_group_recall(subpheno_df_filter, '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/MLP-model/indep.test/12.1/pandisease/', 'owen_profile_mlp_probs.csv', order,'cStage', 1 )
contrast_group_recall['MLP']=MLP_split5_recall_df['mean'].tolist()

for method in ['lr', 'rf', 'xgb']:
    dir = '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/ML-model/indep.test/11.25/pandisease/'
    split_5_recall_df, _ = get_group_recall(subpheno_df_filter, dir,f'owen_profile_{method}_probs.csv', order, 'cStage', 1)
    contrast_group_recall[method] = split_5_recall_df['mean'].tolist()
metaGPT_split5_recall_df, group_counts = get_group_recall(subpheno_df_filter, '/bgi-seq-model-2/codes/zhangkexin/meta_index/output/llama/v4/PanDisease/Split5.specific.aug.Full.sortabu.ema.12.4/',
                                                       'best_ckpt/result/probs/metaGPT.crc.owen.test.prob.csv',
                                                       order, 'cStage', 0)


contrast_group_recall['UmetaGPT']=metaGPT_split5_recall_df['mean'].tolist()
contrast_df = pd.DataFrame(contrast_group_recall, index=order, columns=['MLP', 'lr', 'rf', 'xgb', 'UmetaGPT'])
contrast_df.columns = ['MLP', 'LR', 'RF', 'XGBoost', 'UmetaGPT']
# contrast_df.loc['Average'] = contrast_df.mean(0)
contrast_df.loc['Overall'] = [0.7804, 0.8087, 0.7521, 0.8366, 0.8444]

# group_counts.index = [map_stage[i] for i in group_counts.index]
create_and_plot_heatmap(contrast_df, group_counts,'cStage')

In [ ]:
# 3.25修改
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
# --- 1. 计算函数 (保持原样) ---
def calculate_recall(group_df, threshold=0.5):
    if len(group_df) == 0: return np.nan
    true_positives = pd.Series([True] * len(group_df), index=group_df.index)
    pred_positives = group_df['pred'] == 1
    tp = ((true_positives) & (pred_positives)).sum()
    fn = ((true_positives) & (~pred_positives)).sum()
    if (tp + fn) == 0: return np.nan
    return tp / (tp + fn)
def get_group_recall(subpheno_df, dir, filename, order, object, index_col):
    split_5_recall = {}
    for i in range(1, 6):
        split = 'split' + str(i)
        prob_df = pd.read_csv(os.path.join(dir, split, filename), index_col=index_col)
        prob_459_df = prob_df.loc[list(subpheno_df.index), :]
        prob_459_df[object] = subpheno_df[object]
        location_results = prob_459_df.groupby(object).apply(calculate_recall).reset_index()
        location_results.columns = ['Group', 'Recall']
        location_results['Group'] = pd.Categorical(location_results['Group'], categories=order, ordered=True)
        location_results_sorted = location_results.sort_values('Group')
        split_5_recall[split] = location_results_sorted['Recall'].values.tolist()
    split_5_recall_df = pd.DataFrame(dict(split_5_recall), index=order)
    split_5_recall_df['mean'] = split_5_recall_df.mean(1)
    group_counts = subpheno_df[object].value_counts()
    return split_5_recall_df, group_counts
# --- 2. 核心绘图函数 (去掉顶部标注，标签在右侧) ---
def create_and_plot_heatmap(df, group_counts, object_name):
    # 行归一化 (用于颜色显示)
    data_values_orig = df.values
    data_normalized = data_values_orig.astype('float') / data_values_orig.sum(axis=1)[:, np.newaxis]
    data_normalized = np.nan_to_num(data_normalized)
    models = df.columns
    groups = df.index
    # 配置样式
    plt.rcParams['font.family'] = 'Arial'
    plt.rc('font', size=22)
    plt.rcParams['pdf.fonttype'] = 42
    # 动态调整画布比例
    fig = plt.figure(figsize=(5.5, len(groups) * 1), facecolor='white')
    ax = fig.add_subplot(111)
    # 绘制热图 (颜色使用归一化数据，颜色为 #8AB1D2)
    cmap = sns.blend_palette(["#FFFFFF", "#9DD0C7"], as_cmap=True)
    im = ax.imshow(data_normalized, cmap=cmap, aspect='equal')
    # --- 设置坐标轴 ---
    # 1. 彻底移除顶部和底部的刻度线及标签
    ax.tick_params(top=False, bottom=False, labeltop=False, labelbottom=False)
    # 2. 右侧 Y 轴 (显示分组信息)
    ax.tick_params(left=False, labelleft=False, right=True, labelright=True)
    new_yticklabels = []
    for g in groups:
        if g == 'Overall':
            new_yticklabels.append('Overall')
        else:
            # prefix = "Stage " if object_name == 'cStage' else ""
            prefix = ""
            n_count = group_counts.get(g, 0)
            label = str(int(g)) if isinstance(g, float) else str(g)
            new_yticklabels.append(f"{prefix}{label} (n={n_count})")
    ax.set_yticks(np.arange(len(groups)))
    ax.set_yticklabels(new_yticklabels)
    # --- 在格子里写入原始 Recall 数值 ---
    threshold = (data_normalized.max() + data_normalized.min()) / 2
    for i in range(len(groups)):
        for j in range(len(models)):
            val_orig = data_values_orig[i, j]
            # 文字颜色根据背景深浅自动切换
            text_color = "white" if data_normalized[i, j] > threshold else "black"
            # ax.text(j, i, f"{val_orig:.4f}", ha="center", va="center", color=text_color)
            ax.text(j, i, f"{val_orig:.3f}", ha="center", va="center", color="black", fontsize=18)
    # --- 细节美化 ---
    ax.spines[:].set_visible(False)
    # 白色粗网格线，突出“方块”感
    ax.set_xticks(np.arange(len(models) + 1) - .5, minor=True)
    ax.set_yticks(np.arange(len(groups) + 1) - .5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
    ax.tick_params(which="minor", top=False, bottom=False, left=False, right=False)
    plt.tight_layout()
    plt.savefig(
        f'/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/figures/finetune/11.25/contrast/owen_Recall_{object_name}.4.22.small.pdf',
        bbox_inches='tight')
    plt.show()
# --- 3. 数据处理 (肿瘤位置 & 疾病分期) ---
# 注意：此部分代码需要你的本地环境中有对应的 CSV 文件
contrast_group_recall={}
subpheno_df = pd.read_csv(
    '/bgi-seq-model-2/datasets/zhangkexin/meta_index/metaphlan4/validate/7.25/CRC/owen/CRC_colon_phenotype_Qin.txt',
    sep='\t', index_col=0)
subpheno_df = subpheno_df.drop('T2104103043', errors='ignore')
# 肿瘤位置绘图
order_loc = ['Rectum', 'Sigmoid colon', 'Ascending colon', 'Transverse colon', 'Descending colon', 'Ileoceum',
             'Rectosigmoid junction']
MLP_res, _ = get_group_recall(subpheno_df,
                              '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/MLP-model/indep.test/12.1/pandisease/',
                              'owen_profile_mlp_probs.csv', order_loc, 'Tumor_Location', 1)
contrast_group_recall['MLP'] = MLP_res['mean'].tolist()
for method in ['lr', 'rf', 'xgb']:
    ml_dir = '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/ML-model/indep.test/11.25/pandisease/'
    res, _ = get_group_recall(subpheno_df, ml_dir, f'owen_profile_{method}_probs.csv', order_loc, 'Tumor_Location', 1)
    contrast_group_recall[method.upper()] = res['mean'].tolist()
metaGPT_res, group_counts_loc = get_group_recall(subpheno_df,
                                                 '/bgi-seq-model-2/codes/zhangkexin/meta_index/output/llama/v4/PanDisease/Split5.specific.aug.Full.sortabu.ema.12.4/',
                                                 'best_ckpt/result/probs/metaGPT.crc.owen.test.prob.csv', order_loc,
                                                 'Tumor_Location', 0)
contrast_group_recall['UMetaGPT'] = metaGPT_res['mean'].tolist()
contrast_df_loc = pd.DataFrame(contrast_group_recall, index=order_loc)
contrast_df_loc.loc['Overall'] = [0.7804, 0.8087, 0.7521, 0.8366, 0.8444]
# 这里假设已经得到了 contrast_df_loc 和 group_counts_loc
create_and_plot_heatmap(contrast_df_loc, group_counts_loc, 'Tumor_Location')
# 疾病分期绘图
contrast_group_recall={}
subpheno_df_filter = subpheno_df.dropna(subset=['cStage']).copy()
map_stage = {1.0: 'I', 2.0: 'II', 3.0: 'III', 4.0: 'IV'}
subpheno_df_filter['cStage'] = subpheno_df_filter['cStage'].map(map_stage)
contrast_group_recall_stage = {}
order_stage = ['I', 'II', 'III', 'IV']
MLP_res_s, _ = get_group_recall(subpheno_df_filter,
                                '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/MLP-model/indep.test/12.1/pandisease/',
                                'owen_profile_mlp_probs.csv', order_stage, 'cStage', 1)
contrast_group_recall_stage['MLP'] = MLP_res_s['mean'].tolist()
for method in ['lr', 'rf', 'xgb']:
    ml_dir = '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/ML-model/indep.test/11.25/pandisease/'
    res, _ = get_group_recall(subpheno_df_filter, ml_dir, f'owen_profile_{method}_probs.csv', order_stage, 'cStage', 1)
    contrast_group_recall_stage[method.upper()] = res['mean'].tolist()
metaGPT_res_s, group_counts_stage = get_group_recall(subpheno_df_filter,
                                                     '/bgi-seq-model-2/codes/zhangkexin/meta_index/output/llama/v4/PanDisease/Split5.specific.aug.Full.sortabu.ema.12.4/',
                                                     'best_ckpt/result/probs/metaGPT.crc.owen.test.prob.csv',
                                                     order_stage, 'cStage', 0)
contrast_group_recall_stage['UMetaGPT'] = metaGPT_res_s['mean'].tolist()
contrast_df_stage = pd.DataFrame(contrast_group_recall_stage, index=order_stage)
contrast_df_stage.loc['Overall'] = [0.7804, 0.8087, 0.7521, 0.8366, 0.8444]
# ... (中间数据读取逻辑保持不变)
# 这里假设已经得到了 contrast_df_stage 和 group_counts_stage
create_and_plot_heatmap(contrast_df_stage, group_counts_stage, 'cStage')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
def create_and_plot_heatmap_vertical(df, group_counts, object_name):
    # ------------------ 数据处理部分（保持不变） ------------------
    # 依然按原始行（groups）进行归一化，保证颜色深浅的统计意义不变
    data_values_orig = df.values
    data_normalized = data_values_orig.astype('float') / data_values_orig.sum(axis=1)[:, np.newaxis]
    data_normalized = np.nan_to_num(data_normalized)
    models = df.columns
    groups = df.index
    # 【修改 1】：转置用于画图的矩阵
    # 使模型 (models) 在行 (Y轴)，分组 (groups) 在列 (X轴)
    data_values_orig_T = data_values_orig.T
    data_normalized_T = data_normalized.T
    # 配置样式
    plt.rcParams['font.family'] = 'Arial'
    plt.rc('font', size=22)
    plt.rcParams['pdf.fonttype'] = 42
    # 【修改 2】：动态调整画布比例，宽高逻辑互换
    # 宽度由列数(groups)决定，高度由行数(models)决定
    fig = plt.figure(figsize=(len(groups) * 1, 5.5), facecolor='white')
    ax = fig.add_subplot(111)
    # 【修改 3】：绘制热图传入转置后的数据 data_normalized_T
    cmap = sns.blend_palette(["#FFFFFF", "#9DD0C7"], as_cmap=True)
    im = ax.imshow(data_normalized_T, cmap=cmap, aspect='equal')
    # ------------------ 设置坐标轴 ------------------
    # 【修改 4】：互换 X 轴和 Y 轴的标签逻辑
    # 底部显示 X 轴（Groups），顶部关闭
    ax.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False)
    # 左侧显示 Y 轴（Models），右侧关闭
    ax.tick_params(left=False, labelleft=False, right=False, labelright=False)
    # 4.1 设置 X 轴（原本的 Y 轴逻辑移到这里）
    new_xticklabels = []
    for g in groups:
        if g == 'Overall':
            new_xticklabels.append('Overall')
        else:
            prefix = ""
            n_count = group_counts.get(g, 0)
            label = str(int(g)) if isinstance(g, float) else str(g)
            # 建议加一个换行符 \n 防止标签在底部挤在一起
            new_xticklabels.append(f"{prefix}{label} (n={n_count})")
    ax.set_xticks(np.arange(len(groups)))
    # X 轴标签如果是多行，建议旋转一下以防重叠
    ax.set_xticklabels(new_xticklabels, rotation=90, ha='center')
    # 4.2 设置 Y 轴
    # ax.set_yticks(np.arange(len(models)))
    # ax.set_yticklabels(models)
    # ------------------ 写入数值 ------------------
    # 【修改 5】：循环边界互换，行列索引互换
    threshold = (data_normalized_T.max() + data_normalized_T.min()) / 2
    for i in range(len(models)):  # 行：models (对应转置后的第一维)
        for j in range(len(groups)):  # 列：groups (对应转置后的第二维)
            val_orig = data_values_orig_T[i, j]
            # 文字颜色根据背景深浅自动切换 (需用到转置后的 normalized 数据)
            text_color = "white" if data_normalized_T[i, j] > threshold else "black"
            # 注意 ax.text(x, y, ...)，x 是列索引 j，y 是行索引 i
            ax.text(j, i, f"{val_orig:.3f}", ha="center", va="center", color="black", fontsize=18)
    # ------------------ 细节美化 ------------------
    ax.spines[:].set_visible(False)
    # 网格线边界同样需要互换 len(groups) 和 len(models)
    ax.set_xticks(np.arange(len(groups) + 1) - .5, minor=True)
    ax.set_yticks(np.arange(len(models) + 1) - .5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
    ax.tick_params(which="minor", top=False, bottom=False, left=False, right=False)
    # 避免截断边缘
    plt.tight_layout()
    plt.savefig(
        f'/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/figures/finetune/11.25/contrast/owen_Recall_{object_name}.4.22.vertical.pdf',
        bbox_inches='tight')
    plt.show()
    
contrast_group_recall={}
subpheno_df_filter = subpheno_df.dropna(subset=['cStage']).copy()
map_stage = {1.0: 'I', 2.0: 'II', 3.0: 'III', 4.0: 'IV'}
subpheno_df_filter['cStage'] = subpheno_df_filter['cStage'].map(map_stage)
contrast_group_recall_stage = {}
order_stage = ['I', 'II', 'III', 'IV']
MLP_res_s, _ = get_group_recall(subpheno_df_filter,
                                '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/MLP-model/indep.test/12.1/pandisease/',
                                'owen_profile_mlp_probs.csv', order_stage, 'cStage', 1)
contrast_group_recall_stage['MLP'] = MLP_res_s['mean'].tolist()
for method in ['lr', 'rf', 'xgb']:
    ml_dir = '/bgi-seq-model-2/codes/zhangkexin/meta_index/experiment/results/ML-model/indep.test/11.25/pandisease/'
    res, _ = get_group_recall(subpheno_df_filter, ml_dir, f'owen_profile_{method}_probs.csv', order_stage, 'cStage', 1)
    contrast_group_recall_stage[method.upper()] = res['mean'].tolist()
metaGPT_res_s, group_counts_stage = get_group_recall(subpheno_df_filter,
                                                     '/bgi-seq-model-2/codes/zhangkexin/meta_index/output/llama/v4/PanDisease/Split5.specific.aug.Full.sortabu.ema.12.4/',
                                                     'best_ckpt/result/probs/metaGPT.crc.owen.test.prob.csv',
                                                     order_stage, 'cStage', 0)
contrast_group_recall_stage['UMetaGPT'] = metaGPT_res_s['mean'].tolist()
contrast_df_stage = pd.DataFrame(contrast_group_recall_stage, index=order_stage)
contrast_df_stage.loc['Overall'] = [0.7804, 0.8087, 0.7521, 0.8366, 0.8444]
# ... (中间数据读取逻辑保持不变)
# 这里假设已经得到了 contrast_df_stage 和 group_counts_stage
create_and_plot_heatmap_vertical(contrast_df_stage, group_counts_stage, 'cStage')